In [ ]:
import pandas as pd
import numpy as np
from scripts.plotting import *

X = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_position_matrix.csv"))
Y = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_velocity_matrix.csv"))
# X_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_position_matrix.csv"))
# Y_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_velocity_matrix.csv"))

t = pd.read_csv("./data/s_curve/uniform/s_curve_gt_latent_time_vector.csv")
t = list(t["t"])

In [ ]:
import umap

umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.5, n_components=2, random_state=42)
X_2d = umap_reducer.fit_transform(X)
plot_2d(X_2d, t, "")

In [ ]:
from sklearn.manifold import MDS

mds = MDS(n_components=2, dissimilarity="euclidean", random_state=42)
X_2d = mds.fit_transform(X)

plot_2d(X_2d, t, "")

In [ ]:
from scipy.spatial.distance import cdist
from scipy.spatial import KDTree

In [ ]:
def smooth_values(x, X_2d, k=5, sigma=None):    
    if not sigma:
        tree = KDTree(X_2d)
        dists, _ = tree.query(X_2d, k=k+1)  # k+1 because the first neighbor is the point itself
        avg_dist = np.mean(dists[:, 1:])  # Ignore the 0th distance (self-distance)
        sigma = avg_dist / 10
    
    # Compute pairwise squared distances
    dists = cdist(x, X_2d, metric='sqeuclidean')
    
    # Compute Gaussian weights (no normalization)
    weights = np.exp(-dists / (2 * sigma**2))
    
    # Compute smoothed values (no normalization)
    smoothed_values = np.dot(weights, np.ones(X_2d.shape[0]))
    
    return smoothed_values

In [ ]:
if "X_grid" not in globals():
    n_dim = 2
    density = 1
    grs = []
    for dim_i in range(n_dim):
        m, M = np.min(X_2d[:, dim_i]), np.max(X_2d[:, dim_i])
        m = m - 0.01 * np.abs(M - m)
        M = M + 0.01 * np.abs(M - m)
        gr = np.linspace(m, M, int(50 * density))
        grs.append(gr)
    meshes_tuple = np.meshgrid(*grs)
    X_grid = np.vstack([i.flat for i in meshes_tuple]).T

smoothed_values = smooth_values(X_grid, X_2d)
smoothed_values

In [ ]:
n_dim = 2
density = 1
grs = []
for dim_i in range(n_dim):
    m, M = np.min(X_2d[:, dim_i]), np.max(X_2d[:, dim_i])
    m = m - 0.01 * np.abs(M - m)
    M = M + 0.01 * np.abs(M - m)
    gr = np.linspace(m, M, int(50 * density))
    grs.append(gr)

meshes_tuple = np.meshgrid(*grs)
X_grid = np.vstack([i.flat for i in meshes_tuple]).T

In [ ]:
plt.hist(smooth_values(X_2d, X_2d, sigma=0.1), bins=30, edgecolor='black', alpha=0.7)

In [ ]:
# Create figure and axis
fig, ax = plt.subplots(figsize=(6, 6), facecolor="white", constrained_layout=True)
ax.tick_params(axis='both', which='major', labelsize=8)  # Adjust tick label size

# Add scatter plot with color mapping
add_2d_scatter(ax, X_2d, points_color=smooth_values(X_2d, X_2d, sigma=0.1))

# Set axis ticks
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
plt.show()

In [ ]:
smoothed_values = smooth_values(X_grid, X_2d, sigma=0.01)
threshold = np.percentile(smooth_values(X_2d, X_2d, sigma=0.01), 20)  # Top 10% of values
threshold = 0.1

# Assign colors: white for values below threshold, original for above
colors = np.where(smoothed_values <= threshold, smoothed_values, "white")

# Create figure and axis
fig, ax = plt.subplots(figsize=(6, 6), facecolor="white", constrained_layout=True)
ax.tick_params(axis='both', which='major', labelsize=8)  # Adjust tick label size

# Add scatter plot with color mapping
add_2d_scatter(ax, X_grid, points_color=np.log(smoothed_values))

# Set axis ticks
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

plt.show()

In [ ]:
from scripts.TPS import *

tps = ThinPlateSpline(X_2d, n_control_points=500)
tps.fit(X, dof_target=30)
X_smoothed = tps.predict(X_2d)
plot_3d(X_smoothed,t)

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform

In [ ]:
from scipy.optimize import minimize

def loss(X_flat, D, tps, X, lam):
    # Reshape the 1D array into 2D points (n, 2)
    X_2d = X_flat.reshape((-1, 2))
    X_smoothed = tps.predict(X_2d)
    
    X_norm = np.sum(X_smoothed**2, axis=1, keepdims=True)
    pairwise_sq_dist = X_norm + X_norm.T - 2 * np.dot(X_smoothed, X_smoothed.T)
    
    loss = np.sum((D - pairwise_sq_dist) ** 2) / 2
    
    loss2 = np.sum((X - X_smoothed) ** 2)
    return loss + lam*loss2


def gradient(X_flat, D, tps, X, lam):
    # Reshape the 1D array back into 2D points
    X_2d = X_flat.reshape((-1, 2))
    X_smoothed = tps.predict(X_2d)
    X_norm = np.sum(X_smoothed**2, axis=1, keepdims=True)
    pairwise_sq_dist = X_norm + X_norm.T - 2 * np.dot(X_smoothed, X_smoothed.T)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    
    dist_diff = D - pairwise_sq_dist
    diff = X_smoothed[:, np.newaxis, :] - X_smoothed[np.newaxis, :, :]
    A = np.einsum("nm,nmp -> np", dist_diff, diff)
    grad = -4 * np.einsum("np,npd -> nd", A, jacobians)
    
    grad2 = -2 * np.einsum("nai,na -> ni", jacobians, (X - X_smoothed))
    
    total_grad = grad + lam * grad2
    return total_grad.flatten()

# Optimization function that minimizes the total loss
def optimize_total(tps, X_2d, X, D, lam, disp=False):
    # X here is your initial guess for the 2D points, of shape (n, 2)
    X_flat = X_2d.flatten()  # Flatten for the optimizer
    result = minimize(
        fun=loss,
        x0=X_flat,
        jac=gradient,
        args=(D, tps, X, lam),
        method='L-BFGS-B',
        options={"disp": disp}
    )
    # Reshape the optimized variable back into (n, 2)
    X_optimized = result.x.reshape((-1, 2))
    return X_optimized, result


D = squareform(pdist(X, metric='euclidean')) ** 2 + 1e-3

lam = 10
%time X_optimized, optimization_result = optimize_total(tps, X_2d, X, D, lam, disp=True)

In [ ]:
plot_2d(X_optimized, t, "")

In [ ]:
plot_2d(X_2d, t, "")

In [ ]:
tps = ThinPlateSpline(X_optimized, n_control_points=1000)
tps.fit(X, dof_target=30)
X_smoothed = tps.predict(X_optimized)
plot_3d(X_smoothed,t)

In [ ]:
import numdifftools as nd

X_flat = X_2d[:10,:].flatten()
D = squareform(pdist(X[:10,:], metric='euclidean')) ** 2
loss_func = lambda X_flat: loss(X_flat, D, tps, X[:10,:])
numerical_grad = nd.Gradient(loss_func)(X_flat)
numerical_grad

In [ ]:
manual_grad = gradient(X_flat, D, tps, X[:10,:])
np.linalg.norm(numerical_grad - manual_grad)